In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import folium
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Carregar shapefile
gdf_mt = gpd.read_file('data/external/shapefiles/mt_municipios_2022/MT_Municipios_2022.shp')

# Filtrar Cuiabá e Várzea Grande
MUNICIPIOS = {'5103403': 'Cuiabá', '5108402': 'Várzea Grande'}
gdf_alvo = gdf_mt[gdf_mt['CD_MUN'].isin(MUNICIPIOS.keys())].copy()
gdf_alvo['nome'] = gdf_alvo['CD_MUN'].map(MUNICIPIOS)

print(f"Shapefile carregado: {len(gdf_mt)} municípios MT")
print(f"Municípios alvo: {len(gdf_alvo)}")
print(f"   CRS: {gdf_alvo.crs}")
print(gdf_alvo[['CD_MUN', 'nome', 'AREA_KM2']])

# Carregar dataset de features
df = pd.read_parquet('data/processed/dataset_features_v2.parquet')
df['data'] = pd.to_datetime(df['data'])
print(f"\nDataset features: {df.shape}")
print(f"   Último registro: {df['data'].max().date()}")

DataSourceError: data/external/shapefiles/mt_municipios_2022/MT_Municipios_2022.shp: No such file or directory

In [2]:
from pathlib import Path
for f in Path('data/external/shapefiles/mt_municipios_2022').iterdir():
    print(f.name)

FileNotFoundError: [WinError 3] O sistema não pode encontrar o caminho especificado: 'data\\external\\shapefiles\\mt_municipios_2022'

In [3]:
from pathlib import Path

# Verificar estrutura
base = Path('../data/external/shapefiles')
print("Conteúdo de shapefiles/:")
for f in base.iterdir():
    print(f"  {f.name}/")
    if f.is_dir():
        for sub in f.iterdir():
            print(f"    {sub.name}")

Conteúdo de shapefiles/:
  mt_bairros_2022/
    MT_bairros_CD2022.cpg
    MT_bairros_CD2022.dbf
    MT_bairros_CD2022.prj
    MT_bairros_CD2022.shp
    MT_bairros_CD2022.shx
  MT_bairros_CD2022.zip/
  mt_municipios_2022/
    MT_Municipios_2022.cpg
    MT_Municipios_2022.dbf
    MT_Municipios_2022.prj
    MT_Municipios_2022.shp
    MT_Municipios_2022.shx
  MT_Municipios_2022.zip/
  mt_setores_censitarios.zip/


In [4]:
import pandas as pd
import numpy as np
import geopandas as gpd
import folium
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Carregar shapefile — path relativo ao notebook
gdf_mt = gpd.read_file('../data/external/shapefiles/mt_municipios_2022/MT_Municipios_2022.shp')

# Filtrar Cuiabá e Várzea Grande
MUNICIPIOS = {'5103403': 'Cuiabá', '5108402': 'Várzea Grande'}
gdf_alvo = gdf_mt[gdf_mt['CD_MUN'].isin(MUNICIPIOS.keys())].copy()
gdf_alvo['nome'] = gdf_alvo['CD_MUN'].map(MUNICIPIOS)

print(f"Shapefile: {len(gdf_mt)} municípios MT")
print(f"Alvo: {len(gdf_alvo)} municípios")
print(gdf_alvo[['CD_MUN', 'nome', 'AREA_KM2']])

# Dataset features
df = pd.read_parquet('../data/processed/dataset_features_v2.parquet')
df['data'] = pd.to_datetime(df['data'])
print(f"\nDataset: {df.shape} | Último: {df['data'].max().date()}")

Shapefile: 141 municípios MT
Alvo: 2 municípios
      CD_MUN           nome  AREA_KM2
37   5103403         Cuiabá  4327.448
134  5108402  Várzea Grande   724.279

Dataset: (2242, 55) | Último: 2024-12-28


In [5]:
import requests, zipfile
from pathlib import Path

# URL correta dos setores censitários 2022 — MT
URL_SETORES = (
    'https://geoftp.ibge.gov.br/organizacao_do_territorio/'
    'malhas_territoriais/malhas_de_setores_censitarios__divisoes_intramunicipais/'
    'censo_2022/setores_censitarios_shp/mt/mt_setores_censitarios.zip'
)

# Testar URL antes de baixar
r = requests.head(URL_SETORES, timeout=10, allow_redirects=True)
print(f'Status: {r.status_code}')
print(f'Tamanho: {int(r.headers.get("content-length",0))/1e6:.1f}MB')

Status: 404
Tamanho: 0.0MB


In [6]:
import requests, zipfile
from pathlib import Path

URL_BAIRROS = (
    'https://geoftp.ibge.gov.br/organizacao_do_territorio/'
    'malhas_territoriais/malhas_de_setores_censitarios__divisoes_intramunicipais/'
    'censo_2022/bairros/shp/UF/MT_bairros_CD2022.zip'
)

r = requests.head(URL_BAIRROS, timeout=10, allow_redirects=True)
print(f'Status: {r.status_code}')
print(f'Tamanho: {int(r.headers.get("content-length",0))/1e6:.2f}MB')

Status: 200
Tamanho: 0.56MB


In [7]:
SHAPE_DIR = Path('../data/external/shapefiles')
zip_path = SHAPE_DIR / 'MT_bairros_CD2022.zip'

# Download
print("Baixando bairros MT...")
resp = requests.get(URL_BAIRROS, stream=True, timeout=120)
with open(zip_path, 'wb') as f:
    for chunk in resp.iter_content(chunk_size=8192):
        f.write(chunk)
print(f"Download: {zip_path}")

# Extrair
dest = SHAPE_DIR / 'mt_bairros_2022'
dest.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(dest)
    print(f"Arquivos extraídos:")
    for f in z.namelist():
        print(f"   {f}")

# Carregar e explorar
shp = list(dest.glob('*.shp'))[0]
gdf_bairros = gpd.read_file(shp)
print(f"\nTotal bairros MT: {len(gdf_bairros)}")
print(f"   Colunas: {gdf_bairros.columns.tolist()}")
print(f"   CRS: {gdf_bairros.crs}")
print(gdf_bairros.head(3))

Baixando bairros MT...
Download: ..\data\external\shapefiles\MT_bairros_CD2022.zip
Arquivos extraídos:
   MT_bairros_CD2022.cpg
   MT_bairros_CD2022.dbf
   MT_bairros_CD2022.prj
   MT_bairros_CD2022.shp
   MT_bairros_CD2022.shx

Total bairros MT: 635
   Colunas: ['CD_REGIAO', 'NM_REGIAO', 'CD_UF', 'NM_UF', 'CD_MUN', 'NM_MUN', 'CD_DIST', 'NM_DIST', 'CD_SUBDIST', 'NM_SUBDIST', 'CD_BAIRRO', 'NM_BAIRRO', 'CD_RGINT', 'NM_RGINT', 'CD_RGI', 'NM_RGI', 'CD_CONCURB', 'NM_CONCURB', 'geometry']
   CRS: EPSG:4674
  CD_REGIAO     NM_REGIAO CD_UF        NM_UF   CD_MUN    NM_MUN    CD_DIST  \
0         5  Centro-Oeste    51  Mato Grosso  5100201  Água Boa  510020105   
1         5  Centro-Oeste    51  Mato Grosso  5100201  Água Boa  510020105   
2         5  Centro-Oeste    51  Mato Grosso  5100201  Água Boa  510020105   

    NM_DIST   CD_SUBDIST NM_SUBDIST   CD_BAIRRO  NM_BAIRRO CD_RGINT  \
0  Água Boa  51002010500       None  5100201009  Centro II     5104   
1  Água Boa  51002010500       None  51

In [8]:
# Filtrar Cuiabá e Várzea Grande
gdf_bairros_alvo = gdf_bairros[gdf_bairros['CD_MUN'].isin(['5103403', '5108402'])].copy()

print(f"Bairros filtrados: {len(gdf_bairros_alvo)}")
print(f"\nPor município:")
print(gdf_bairros_alvo.groupby(['NM_MUN'])['NM_BAIRRO'].count())
print(f"\nBairros de Cuiabá:")
print(sorted(gdf_bairros_alvo[gdf_bairros_alvo['CD_MUN']=='5103403']['NM_BAIRRO'].tolist()))

Bairros filtrados: 143

Por município:
NM_MUN
Cuiabá           119
Várzea Grande     24
Name: NM_BAIRRO, dtype: int64

Bairros de Cuiabá:
['Alto da Serra I', 'Altos do Coxipó', 'Alvorada', 'Araés', 'Areão', 'Bandeirantes', 'Barbado', 'Barra do Pari', 'Baú', 'Bela Marina', 'Bela Vista', 'Boa Esperança', 'Bosque da Saúde', 'Cachoeira das Garças', 'Campo Velho', 'Campo Verde', 'Canjica', 'Carumbé', 'Centro Norte', 'Centro Político Administrativo', 'Centro Sul', 'Chácara dos Pinheiros', 'Cidade Alta', 'Cidade Verde', 'Cohab São Gonçalo', 'Coophamil', 'Coophema', 'Coxipó', 'Despraiado', 'Dom Aquino', 'Dom Bosco', 'Dr. Fábio Leite I', 'Duque de Caxias', 'Goiabeira', 'Grande Terceiro', 'Jardim Aclimação', 'Jardim Califórnia', 'Jardim Campus Elizius', 'Jardim Comodoro', 'Jardim Cuiabá', 'Jardim Eldorado', 'Jardim Europa', 'Jardim Florianópolis', 'Jardim Fortaleza', 'Jardim Gramado', 'Jardim Imperial', 'Jardim Industriário', 'Jardim Itália', 'Jardim Leblon', 'Jardim Mariana', 'Jardim Mossoró', 

In [9]:
import lightgbm as lgb
import pickle
import warnings
warnings.filterwarnings('ignore')

# ── 1. Previsão do modelo para os últimos dados disponíveis ──────────────────
# Usar as últimas 4 semanas do dataset como base da previsão
df_sorted = df.sort_values('data')
ultimos = df_sorted.tail(28).copy()  # últimos 28 dias

# Features usadas no modelo (remover colunas não-feature)
colunas_excluir = ['data', 'casos', 'ano', 'mes', 'semana']
colunas_excluir = [c for c in colunas_excluir if c in ultimos.columns]
features = [c for c in ultimos.columns if c not in colunas_excluir]

print(f"Features disponíveis: {len(features)}")
print(f"   Período base: {ultimos['data'].min().date()} → {ultimos['data'].max().date()}")

# Estatísticas dos últimos 28 dias
casos_recentes = df_sorted.tail(28)['casos'].sum()
casos_media_diaria = df_sorted.tail(28)['casos'].mean()
casos_previstos_semana = casos_media_diaria * 7  # proxy da previsão semanal

print(f"\nCasos últimos 28 dias: {casos_recentes:.0f}")
print(f"   Média diária: {casos_media_diaria:.1f}")
print(f"   Previsão proxy próxima semana: {casos_previstos_semana:.0f} casos")

# ── 2. Distribuição por bairro via densidade populacional ────────────────────
# Calcular área de cada bairro como proxy de população
# (Censo 2022 não tem pop por bairro direto — usamos área como proxy)
gdf_bairros_alvo = gdf_bairros_alvo.to_crs('EPSG:32721')  # projeção métrica MT
gdf_bairros_alvo['area_km2'] = gdf_bairros_alvo.geometry.area / 1e6
gdf_bairros_alvo = gdf_bairros_alvo.to_crs('EPSG:4674')   # volta ao geográfico

# Pop por município (Censo 2022)
pop_municipios = {
    '5103403': 651_098,  # Cuiabá
    '5108402': 287_526,  # Várzea Grande
}

# Área total por município
area_total = gdf_bairros_alvo.groupby('CD_MUN')['area_km2'].sum().to_dict()

# Casos previstos por município (proporcional à população)
pop_total = sum(pop_municipios.values())
casos_por_mun = {
    cd: casos_previstos_semana * (pop / pop_total)
    for cd, pop in pop_municipios.items()
}

print(f"\nCasos previstos por município:")
for cd, casos in casos_por_mun.items():
    nome = {'5103403': 'Cuiabá', '5108402': 'Várzea Grande'}[cd]
    print(f"   {nome}: {casos:.1f} casos/semana")

Features disponíveis: 51
   Período base: 2024-12-01 → 2024-12-28

Casos últimos 28 dias: 1687
   Média diária: 60.2
   Previsão proxy próxima semana: 422 casos

Casos previstos por município:
   Cuiabá: 292.6 casos/semana
   Várzea Grande: 129.2 casos/semana


In [24]:
import numpy as np

# ── 3. Score de risco por bairro ─────────────────────────────────────────────
# Fórmula: Score = Casos_mun × (Area_bairro / Area_mun) × Fator_densidade
# Referência: Nature Commun 2022 — densidade populacional driver dominante dengue

def calcular_score(row):
    cd_mun = row['CD_MUN']
    area_mun = area_total.get(cd_mun, 1)
    casos_mun = casos_por_mun.get(cd_mun, 0)
    
    # Casos esperados proporcional à área
    casos_bairro = casos_mun * (row['area_km2'] / area_mun)
    
    # Fator densidade (bairros menores e mais densos = maior risco)
    # Pop estimada do bairro = Pop_mun × (Area_bairro / Area_mun)
    pop_mun = pop_municipios.get(cd_mun, 1)
    pop_bairro_est = pop_mun * (row['area_km2'] / area_mun)
    densidade = pop_bairro_est / row['area_km2'] if row['area_km2'] > 0 else 0
    
    return casos_bairro, pop_bairro_est, densidade

resultado = gdf_bairros_alvo.apply(calcular_score, axis=1, result_type='expand')
resultado.columns = ['casos_esperados', 'pop_estimada', 'densidade_hab_km2']
gdf_bairros_alvo = gdf_bairros_alvo.join(resultado)

# Incidência por 100k hab (padrão epidemiológico)
gdf_bairros_alvo['incidencia_100k'] = (
    gdf_bairros_alvo['casos_esperados'] / 
    gdf_bairros_alvo['pop_estimada'] * 100_000
)

# Score normalizado 0-1
max_inc = gdf_bairros_alvo['incidencia_100k'].max()
gdf_bairros_alvo['score'] = gdf_bairros_alvo['incidencia_100k'] / max_inc

# Classificação de risco
def classificar_risco(score, pop):
    if pop < 500:
        return 'Baixa Confiança'
    elif score >= 0.75:
        return 'Alto'
    elif score >= 0.50:
        return 'Moderado'
    elif score >= 0.25:
        return 'Baixo'
    else:
        return 'Muito Baixo'

gdf_bairros_alvo['risco'] = gdf_bairros_alvo.apply(
    lambda r: classificar_risco(r['score'], r['pop_estimada']), axis=1
)

print("Score calculado!")
print(f"\nDistribuição de risco:")
print(gdf_bairros_alvo['risco'].value_counts())
print(f"\nTop 10 bairros maior risco:")
cols = ['NM_BAIRRO', 'NM_MUN', 'casos_esperados', 'incidencia_100k', 'score', 'risco']
print(gdf_bairros_alvo.nlargest(10, 'score')[cols].to_string())

Score calculado!

Distribuição de risco:
risco
Alto               141
Baixa Confiança      2
Name: count, dtype: int64

Top 10 bairros maior risco:
            NM_BAIRRO  NM_MUN  casos_esperados  incidencia_100k  score risco
170          Pedregal  Cuiabá         1.237897        44.932795    1.0  Alto
171    Jardim Vitória  Cuiabá         2.841352        44.932795    1.0  Alto
172   Jardim Tropical  Cuiabá         0.635345        44.932795    1.0  Alto
173     Jardim Cuiabá  Cuiabá         1.420426        44.932795    1.0  Alto
174  Jardim Ubirajara  Cuiabá         0.631838        44.932795    1.0  Alto
175       Cidade Alta  Cuiabá         4.420374        44.932795    1.0  Alto
176        Dom Aquino  Cuiabá         3.572171        44.932795    1.0  Alto
177           Popular  Cuiabá         0.516866        44.932795    1.0  Alto
178               Baú  Cuiabá         0.856115        44.932795    1.0  Alto
180     Jardim Leblon  Cuiabá         1.471187        44.932795    1.0  Alto


In [25]:
import requests

# API IBGE — agregados por bairro Censo 2022
# Variável 6318 = Domicílios particulares ocupados
def buscar_domicilios_ibge(cod_municipio):
    url = (
        f"https://servicodados.ibge.gov.br/api/v3/agregados/9514/periodos/2022/"
        f"variaveis/6318?localidades=N315[N6[{cod_municipio}]]"
    )
    try:
        r = requests.get(url, timeout=15)
        data = r.json()
        print(f"Status: {r.status_code}")
        print(data[0]['resultados'][0]['series'][:3])
    except Exception as e:
        print(f"Erro: {e}")

print("Testando API IBGE para Cuiabá...")
buscar_domicilios_ibge('5103403')

Testando API IBGE para Cuiabá...
Status: 500
Erro: 0


In [26]:
# Solução: usar área + fator de urbanização via centróide do bairro
# Bairros com área menor = mais urbanizados = maior densidade real
# Isso é demograficamente mais correto para cidades brasileiras

gdf_bairros_alvo = gdf_bairros_alvo.to_crs('EPSG:32721')

# Densidade inversa à área — bairros menores são mais densos em cidades
# Referência: padrão de urbanização horizontal brasileiro
area_max = gdf_bairros_alvo.groupby('CD_MUN')['area_km2'].transform('max')
area_min = gdf_bairros_alvo.groupby('CD_MUN')['area_km2'].transform('min')

# Fator de densidade: bairros menores = mais densos (padrão urbano BR)
gdf_bairros_alvo['fator_densidade'] = 1 - (
    (gdf_bairros_alvo['area_km2'] - area_min) / 
    (area_max - area_min + 1e-9)
)

# Recalcular casos com fator de densidade
# Normalizar fator para soma = 1 por município
soma_fatores = gdf_bairros_alvo.groupby('CD_MUN')['fator_densidade'].transform('sum')
gdf_bairros_alvo['peso_normalizado'] = gdf_bairros_alvo['fator_densidade'] / soma_fatores

# Casos esperados por bairro
gdf_bairros_alvo['casos_esperados_v2'] = gdf_bairros_alvo.apply(
    lambda r: casos_por_mun.get(r['CD_MUN'], 0) * r['peso_normalizado'], axis=1
)

# Pop estimada proporcional ao peso (não à área)
gdf_bairros_alvo['pop_estimada_v2'] = gdf_bairros_alvo.apply(
    lambda r: pop_municipios.get(r['CD_MUN'], 0) * r['peso_normalizado'], axis=1
)

# Incidência por 100k
gdf_bairros_alvo['incidencia_100k_v2'] = (
    gdf_bairros_alvo['casos_esperados_v2'] /
    gdf_bairros_alvo['pop_estimada_v2'] * 100_000
)

# Score normalizado
max_inc = gdf_bairros_alvo['incidencia_100k_v2'].max()
min_inc = gdf_bairros_alvo['incidencia_100k_v2'].min()
gdf_bairros_alvo['score_v2'] = (
    (gdf_bairros_alvo['incidencia_100k_v2'] - min_inc) /
    (max_inc - min_inc + 1e-9)
)

# Classificação
gdf_bairros_alvo['risco_v2'] = gdf_bairros_alvo.apply(
    lambda r: classificar_risco(r['score_v2'], r['pop_estimada_v2']), axis=1
)

gdf_bairros_alvo = gdf_bairros_alvo.to_crs('EPSG:4674')

print("Score v2 calculado!")
print(f"\nDistribuição de risco:")
print(gdf_bairros_alvo['risco_v2'].value_counts())
print(f"\nTop 10 bairros maior risco:")
cols = ['NM_BAIRRO', 'NM_MUN', 'area_km2', 'casos_esperados_v2', 'score_v2', 'risco_v2']
print(gdf_bairros_alvo.nlargest(10, 'score_v2')[cols].to_string())

Score v2 calculado!

Distribuição de risco:
risco_v2
Muito Baixo        141
Baixa Confiança      2
Name: count, dtype: int64

Top 10 bairros maior risco:
                          NM_BAIRRO  NM_MUN  area_km2  casos_esperados_v2  score_v2     risco_v2
170                        Pedregal  Cuiabá  0.624582            2.701949  0.000007  Muito Baixo
171                  Jardim Vitória  Cuiabá  1.433607            2.382075  0.000007  Muito Baixo
172                 Jardim Tropical  Cuiabá  0.320564            2.822152  0.000007  Muito Baixo
173                   Jardim Cuiabá  Cuiabá  0.716677            2.665536  0.000007  Muito Baixo
174                Jardim Ubirajara  Cuiabá  0.318795            2.822852  0.000007  Muito Baixo
176                      Dom Aquino  Cuiabá  1.802342            2.236283  0.000007  Muito Baixo
179  Centro Político Administrativo  Cuiabá  6.259260            0.474097  0.000007  Muito Baixo
180                   Jardim Leblon  Cuiabá  0.742289            2.655

In [27]:
import requests
import pandas as pd

# API CNES — estabelecimentos de saúde por município
def buscar_cnes_municipio(cod_municipio, nome):
    url = f"https://apidadosabertos.saude.gov.br/cnes/estabelecimentos?codigo_municipio={cod_municipio}&limit=100&offset=0"
    try:
        r = requests.get(url, timeout=15)
        print(f"\n{nome} — Status: {r.status_code}")
        if r.status_code == 200:
            data = r.json()
            print(f"  Chaves: {list(data.keys())}")
            print(f"  Amostra: {str(data)[:300]}")
        else:
            print(f"  Erro: {r.text[:200]}")
    except Exception as e:
        print(f"  Exceção: {e}")

buscar_cnes_municipio('510340', 'Cuiabá')
buscar_cnes_municipio('510840', 'Várzea Grande')


Cuiabá — Status: 200
  Chaves: ['estabelecimentos']
  Amostra: {'estabelecimentos': [{'codigo_cnes': 9635890, 'numero_cnpj_entidade': None, 'nome_razao_social': 'LABORATORIO CARLOS CHAGAS LTDA', 'nome_fantasia': 'POSTO DE COLETA UNIDADE ONCOMED', 'natureza_organizacao_entidade': None, 'tipo_gestao': 'M', 'descricao_nivel_hierarquia': None, 'descricao_esfera_adm

Várzea Grande — Status: 200
  Chaves: ['estabelecimentos']
  Amostra: {'estabelecimentos': [{'codigo_cnes': 9640290, 'numero_cnpj_entidade': None, 'nome_razao_social': 'MARIANA LIMA PEREIRA ODONTOLOGIA', 'nome_fantasia': 'MARIANA LIMA PEREIRA ODONTOLOGIA', 'natureza_organizacao_entidade': None, 'tipo_gestao': 'M', 'descricao_nivel_hierarquia': None, 'descricao_esfera_


In [28]:
def buscar_todos_cnes(cod_municipio, nome):
    todos = []
    offset = 0
    limit = 100
    
    while True:
        url = (f"https://apidadosabertos.saude.gov.br/cnes/estabelecimentos"
               f"?codigo_municipio={cod_municipio}&limit={limit}&offset={offset}")
        r = requests.get(url, timeout=15)
        data = r.json()
        estabelecimentos = data.get('estabelecimentos', [])
        if not estabelecimentos:
            break
        todos.extend(estabelecimentos)
        offset += limit
        if len(estabelecimentos) < limit:
            break
    
    df = pd.DataFrame(todos)
    print(f"\n{nome}: {len(df)} estabelecimentos")
    print(f"Colunas disponíveis: {df.columns.tolist()}")
    return df

df_cuiaba = buscar_todos_cnes('510340', 'Cuiabá')
df_vg = buscar_todos_cnes('510840', 'Várzea Grande')


Cuiabá: 20 estabelecimentos
Colunas disponíveis: ['codigo_cnes', 'numero_cnpj_entidade', 'nome_razao_social', 'nome_fantasia', 'natureza_organizacao_entidade', 'tipo_gestao', 'descricao_nivel_hierarquia', 'descricao_esfera_administrativa', 'codigo_tipo_unidade', 'codigo_cep_estabelecimento', 'endereco_estabelecimento', 'numero_estabelecimento', 'bairro_estabelecimento', 'numero_telefone_estabelecimento', 'latitude_estabelecimento_decimo_grau', 'longitude_estabelecimento_decimo_grau', 'endereco_email_estabelecimento', 'numero_cnpj', 'codigo_identificador_turno_atendimento', 'descricao_turno_atendimento', 'estabelecimento_faz_atendimento_ambulatorial_sus', 'codigo_estabelecimento_saude', 'codigo_uf', 'codigo_municipio', 'descricao_natureza_juridica_estabelecimento', 'codigo_motivo_desabilitacao_estabelecimento', 'estabelecimento_possui_centro_cirurgico', 'estabelecimento_possui_centro_obstetrico', 'estabelecimento_possui_centro_neonatal', 'estabelecimento_possui_atendimento_hospitalar',

In [29]:
# Verificar coordenadas e quantos têm lat/lon
df_cnes = pd.concat([df_cuiaba, df_vg], ignore_index=True)

print(f"Total estabelecimentos: {len(df_cnes)}")
print(f"\nCom latitude: {df_cnes['latitude_estabelecimento_decimo_grau'].notna().sum()}")
print(f"Com longitude: {df_cnes['longitude_estabelecimento_decimo_grau'].notna().sum()}")

# Amostra das coordenadas
cols = ['nome_fantasia', 'bairro_estabelecimento', 
        'latitude_estabelecimento_decimo_grau', 
        'longitude_estabelecimento_decimo_grau',
        'codigo_municipio']
print(f"\nAmostra:")
print(df_cnes[cols].head(10).to_string())

# Verificar se há mais páginas — testar offset=100
url_test = ("https://apidadosabertos.saude.gov.br/cnes/estabelecimentos"
            "?codigo_municipio=510340&limit=100&offset=100")
r = requests.get(url_test, timeout=15)
data = r.json()
print(f"\nPágina 2 Cuiabá: {len(data.get('estabelecimentos', []))} registros")

Total estabelecimentos: 40

Com latitude: 40
Com longitude: 40

Amostra:
                            nome_fantasia bairro_estabelecimento  latitude_estabelecimento_decimo_grau  longitude_estabelecimento_decimo_grau  codigo_municipio
0         POSTO DE COLETA UNIDADE ONCOMED             CENTRO SUL                            -15.596000                             -56.097000            510340
1                               FEB SAUDE        BOSQUE DA SAUDE                            -15.596000                             -56.097000            510340
2                 CENTRO DE RETINA CUIABA    JARDIM DAS AMERICAS                            -15.609996                             -56.072791            510340
3          LABORATORIO CARLOS CHAGAS LTDA        BOSQUE DA SAUDE                            -15.589834                             -56.075609            510340
4           POSTO DE COLETA UNIDADE IRHPA           BANDEIRANTES                            -15.596000                         

In [30]:
def buscar_todos_cnes_paginado(cod_municipio, nome):
    todos = []
    offset = 0
    limit = 100
    
    while True:
        url = (f"https://apidadosabertos.saude.gov.br/cnes/estabelecimentos"
               f"?codigo_municipio={cod_municipio}&limit={limit}&offset={offset}")
        r = requests.get(url, timeout=15)
        data = r.json()
        lote = data.get('estabelecimentos', [])
        if not lote:
            break
        todos.extend(lote)
        print(f"  {nome}: offset={offset} → {len(lote)} registros")
        offset += limit
        if len(lote) < limit:
            break
    
    return pd.DataFrame(todos)

print("Buscando todos os estabelecimentos...")
df_cuiaba_full = buscar_todos_cnes_paginado('510340', 'Cuiabá')
df_vg_full = buscar_todos_cnes_paginado('510840', 'Várzea Grande')

df_cnes_full = pd.concat([df_cuiaba_full, df_vg_full], ignore_index=True)
print(f"\nTotal geral: {len(df_cnes_full)}")
print(f"\nTipos de unidade (codigo_tipo_unidade):")
print(df_cnes_full['codigo_tipo_unidade'].value_counts().head(15))

Buscando todos os estabelecimentos...
  Cuiabá: offset=0 → 20 registros
  Várzea Grande: offset=0 → 20 registros

Total geral: 40

Tipos de unidade (codigo_tipo_unidade):
codigo_tipo_unidade
22    16
36     9
39     8
42     1
43     1
77     1
62     1
2      1
73     1
1      1
Name: count, dtype: int64


In [31]:
# Verificar o que a API retorna sobre total
url = ("https://apidadosabertos.saude.gov.br/cnes/estabelecimentos"
       "?codigo_municipio=510340&limit=100&offset=0")
r = requests.get(url, timeout=15)
data = r.json()
print(f"Chaves da resposta: {list(data.keys())}")
print(f"Total na resposta: {len(data.get('estabelecimentos', []))}")

# Verificar offset=20, 40, 60...
for offset in [20, 40, 60, 80, 100]:
    url = (f"https://apidadosabertos.saude.gov.br/cnes/estabelecimentos"
           f"?codigo_municipio=510340&limit=100&offset={offset}")
    r = requests.get(url, timeout=15)
    data = r.json()
    n = len(data.get('estabelecimentos', []))
    print(f"  offset={offset}: {n} registros")
    if n == 0:
        break

Chaves da resposta: ['estabelecimentos']
Total na resposta: 20
  offset=20: 20 registros
  offset=40: 20 registros
  offset=60: 20 registros
  offset=80: 20 registros
  offset=100: 20 registros


In [32]:
# Buscar todas as páginas até acabar
def buscar_todas_paginas(cod_municipio, nome):
    todos = []
    offset = 0
    
    while True:
        url = (f"https://apidadosabertos.saude.gov.br/cnes/estabelecimentos"
               f"?codigo_municipio={cod_municipio}&limit=20&offset={offset}")
        r = requests.get(url, timeout=15)
        lote = r.json().get('estabelecimentos', [])
        if not lote:
            break
        todos.extend(lote)
        offset += 20
        print(f"\r  {nome}: {offset} registros buscados...", end='')
    
    print(f"\n  Total {nome}: {len(todos)}")
    return pd.DataFrame(todos)

df_cuiaba_full = buscar_todas_paginas('510340', 'Cuiabá')
df_vg_full = buscar_todas_paginas('510840', 'Várzea Grande')

df_cnes_full = pd.concat([df_cuiaba_full, df_vg_full], ignore_index=True)

# Tipos de unidade
print(f"\nTotal geral: {len(df_cnes_full)}")
print(f"\nTipos (codigo_tipo_unidade):")
print(df_cnes_full['codigo_tipo_unidade'].value_counts())

  Cuiabá: 2740 registros buscados...
  Total Cuiabá: 2727
  Várzea Grande: 380 registros buscados...
  Total Várzea Grande: 372

Total geral: 3099

Tipos (codigo_tipo_unidade):
codigo_tipo_unidade
22    1810
36     596
39     250
2      131
43     106
42      43
5       37
77      19
60      16
40      10
7        9
70       8
4        7
73       7
62       6
85       6
68       5
69       4
71       4
84       3
81       3
72       3
1        3
75       3
80       2
50       2
76       1
82       1
83       1
61       1
20       1
74       1
Name: count, dtype: int64


In [33]:
# Tipos relevantes para notificação de dengue:
# 1 = Posto de Saúde
# 2 = Centro de Saúde / UBS  
# 5 = PSF (Programa Saúde da Família / ESF)
# 7 = Hospital Geral
# 36 = Clínica/Centro de Especialidade
# 39 = Unidade de Apoio/Diagnose/Terapia (SADT)

# Filtrar apenas unidades notificadoras de dengue
TIPOS_NOTIFICADORES = [1, 2, 5, 7, 20, 40, 60, 68, 69, 70, 71, 72]

df_notif = df_cnes_full[
    df_cnes_full['codigo_tipo_unidade'].isin(TIPOS_NOTIFICADORES)
].copy()

print(f"Unidades notificadoras: {len(df_notif)}")
print(f"\nPor tipo:")
for tipo, grupo in df_notif.groupby('codigo_tipo_unidade'):
    print(f"  Tipo {tipo}: {len(grupo)} unidades — ex: {grupo['nome_fantasia'].iloc[0]}")

print(f"\nCom coordenadas válidas:")
df_notif = df_notif[
    df_notif['latitude_estabelecimento_decimo_grau'].notna() &
    (df_notif['latitude_estabelecimento_decimo_grau'] != 0)
].copy()
print(f"  {len(df_notif)} unidades com lat/lon")
print(f"\nAmostra:")
cols = ['nome_fantasia', 'bairro_estabelecimento', 
        'latitude_estabelecimento_decimo_grau',
        'longitude_estabelecimento_decimo_grau']
print(df_notif[cols].head(10).to_string())

Unidades notificadoras: 231

Por tipo:
  Tipo 1: 3 unidades — ex: UNIDADE RURAL COIVARAS
  Tipo 2: 131 unidades — ex: USF JOCKEY CLUB
  Tipo 5: 37 unidades — ex: HOSPITAL ESTADUAL SANTA CASA
  Tipo 7: 9 unidades — ex: SUPREME CARE MEDICINA INTENSIVA
  Tipo 20: 1 unidades — ex: UNIDADE AVANCADA SANTA ROSA
  Tipo 40: 10 unidades — ex: TAIAMA EMERGENCIAS MEDICAS
  Tipo 60: 16 unidades — ex: SANTA ROSA COOP
  Tipo 68: 5 unidades — ex: CISVARC
  Tipo 69: 4 unidades — ex: DIAGNOSTICOS LABORATORIO SANTA ROSA TOWER
  Tipo 70: 8 unidades — ex: CAPS CPA IV
  Tipo 71: 4 unidades — ex: NUCLEO DE APOIO A SAUDE DA FAMILIA NASF OESTE I
  Tipo 72: 3 unidades — ex: CASAI CUIABA

Com coordenadas válidas:
  219 unidades com lat/lon

Amostra:
                                       nome_fantasia bairro_estabelecimento  latitude_estabelecimento_decimo_grau  longitude_estabelecimento_decimo_grau
98         DIAGNOSTICOS LABORATORIO SANTA ROSA TOWER      RIBEIRAO DA PONTE                            -15.579183 

In [34]:
# Carregar Silver SINAN para cruzar com ID_UNIDADE
import os
from pathlib import Path

# Carregar todos os anos do Silver SINAN
sinan_dir = Path('../data/silver/sinan')
dfs_sinan = []

for arq in sorted(sinan_dir.glob('*.parquet')):
    df_ano = pd.read_parquet(arq)
    dfs_sinan.append(df_ano)
    print(f"  {arq.name}: {len(df_ano):,} registros | colunas: {df_ano.columns.tolist()}")

df_sinan = pd.concat(dfs_sinan, ignore_index=True)
print(f"\nTotal SINAN: {len(df_sinan):,} registros")
print(f"Colunas: {df_sinan.columns.tolist()}")

  dengue_mt_2007.parquet: 14,357 registros | colunas: ['DT_NOTIFIC', 'SG_UF_NOT', 'ID_MUNICIP', 'CS_SEXO', 'NU_IDADE_N', 'HOSPITALIZ', 'EVOLUCAO', 'ano']
  dengue_mt_2008.parquet: 5,871 registros | colunas: ['DT_NOTIFIC', 'SG_UF_NOT', 'ID_MUNICIP', 'CS_SEXO', 'NU_IDADE_N', 'HOSPITALIZ', 'EVOLUCAO', 'ano']
  dengue_mt_2009.parquet: 49,115 registros | colunas: ['DT_NOTIFIC', 'SG_UF_NOT', 'ID_MUNICIP', 'CS_SEXO', 'NU_IDADE_N', 'HOSPITALIZ', 'EVOLUCAO', 'ano']
  dengue_mt_2010.parquet: 35,818 registros | colunas: ['DT_NOTIFIC', 'SG_UF_NOT', 'ID_MUNICIP', 'CS_SEXO', 'NU_IDADE_N', 'HOSPITALIZ', 'EVOLUCAO', 'ano']
  dengue_mt_2011.parquet: 5,097 registros | colunas: ['DT_NOTIFIC', 'SG_UF_NOT', 'ID_MUNICIP', 'CS_SEXO', 'NU_IDADE_N', 'HOSPITALIZ', 'EVOLUCAO', 'ano']
  dengue_mt_2012.parquet: 29,345 registros | colunas: ['DT_NOTIFIC', 'SG_UF_NOT', 'ID_MUNICIP', 'CS_SEXO', 'NU_IDADE_N', 'HOSPITALIZ', 'EVOLUCAO', 'ano']
  dengue_mt_2013.parquet: 31,876 registros | colunas: ['DT_NOTIFIC', 'SG_UF_NO

In [35]:
# Verificar Bronze — arquivo 2024 como amostra
import pandas as pd
from pathlib import Path

df_bronze = pd.read_parquet('../data/bronze/sinan/dengue_mt_2024.parquet')
print(f"Bronze 2024: {df_bronze.shape}")

# Verificar se ID_UNIDADE existe
colunas_interesse = [c for c in df_bronze.columns 
                     if any(x in c.upper() for x in ['UNIDADE', 'BAIRRO', 'CEP', 'LOGRA'])]
print(f"\nColunas geográficas disponíveis:")
for c in colunas_interesse:
    nao_nulos = df_bronze[c].notna().sum()
    pct = nao_nulos/len(df_bronze)*100
    print(f"  {c}: {nao_nulos:,} não-nulos ({pct:.0f}%)")

Bronze 2024: (42742, 121)

Colunas geográficas disponíveis:
  ID_UNIDADE: 42,742 não-nulos (100%)


In [37]:
import pyarrow.parquet as pq

bronze_dir = Path('../data/bronze/sinan')

for arq in sorted(bronze_dir.glob('*.parquet')):
    # Ler apenas schema (sem carregar dados)
    schema = pq.read_schema(arq)
    colunas = schema.names
    
    if 'ID_UNIDADE' in colunas:
        df = pd.read_parquet(arq, columns=['ID_UNIDADE'])
        pct = df['ID_UNIDADE'].notna().sum() / len(df) * 100
        print(f"  {arq.stem}: {pct:.0f}% preenchido ({df['ID_UNIDADE'].notna().sum():,})")
    else:
        print(f"  {arq.stem}: não existe")

  dengue_mt_2007: 100% preenchido (14,357)
  dengue_mt_2008: 100% preenchido (5,871)
  dengue_mt_2009: 100% preenchido (49,115)
  dengue_mt_2010: 100% preenchido (35,818)
  dengue_mt_2011: 100% preenchido (5,097)
  dengue_mt_2012: 100% preenchido (29,345)
  dengue_mt_2013: 100% preenchido (31,876)
  dengue_mt_2014: 100% preenchido (6,432)
  dengue_mt_2015: 100% preenchido (19,288)
  dengue_mt_2016: 100% preenchido (17,815)
  dengue_mt_2017: 100% preenchido (8,509)
  dengue_mt_2018: 100% preenchido (10,131)
  dengue_mt_2019: 100% preenchido (17,855)
  dengue_mt_2020: 100% preenchido (47,631)
  dengue_mt_2021: 100% preenchido (34,174)
  dengue_mt_2022: 100% preenchido (35,341)
  dengue_mt_2023: 100% preenchido (28,605)
  dengue_mt_2024: 100% preenchido (42,742)


In [11]:
import pandas as pd
import numpy as np
import geopandas as gpd
import requests
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Paths
sinan_dir = Path('../data/silver/sinan')
SHAPE_DIR = Path('../data/external/shapefiles')

# Recarregar Silver SINAN
dfs_sinan = []
for arq in sorted(sinan_dir.glob('*.parquet')):
    dfs_sinan.append(pd.read_parquet(arq))
df_sinan = pd.concat(dfs_sinan, ignore_index=True)
print(f"✅ Silver SINAN: {df_sinan.shape}")

# Filtrar Cuiabá (510340) e Várzea Grande (510790)
df_sinan['ID_MUNICIP'] = df_sinan['ID_MUNICIP'].astype(str).str.strip()
df_local = df_sinan[df_sinan['ID_MUNICIP'].isin(['510340', '510790'])].copy()
print(f"✅ Casos locais: {len(df_local):,}")
print(f"   Colunas: {df_local.columns.tolist()}")

# Carga histórica por unidade
df_local['ID_UNIDADE'] = df_local['ID_UNIDADE'].astype(str).str.strip()
carga_unidade = (df_local.groupby('ID_UNIDADE')
                 .size()
                 .reset_index(name='casos_historicos')
                 .sort_values('casos_historicos', ascending=False))

print(f"\n✅ Unidades notificadoras únicas: {len(carga_unidade)}")
print(f"\nTop 15:")
print(carga_unidade.head(15).to_string())

✅ Silver SINAN: (390048, 9)
✅ Casos locais: 88,550
   Colunas: ['DT_NOTIFIC', 'SG_UF_NOT', 'ID_MUNICIP', 'ID_UNIDADE', 'CS_SEXO', 'NU_IDADE_N', 'HOSPITALIZ', 'EVOLUCAO', 'ano']

✅ Unidades notificadoras únicas: 242

Top 15:
    ID_UNIDADE  casos_historicos
203    7099371             10721
44     2495015              7574
31     2471019              6997
153    4070232              6139
111    2795760              5353
30     2470993              4868
188    6296858              3636
110    2795671              3285
35     2471086              3226
81     2655004              2149
26     2470950              2076
150    3953238              1732
78     2604388              1551
103    2674955              1501
172    6085423              1227


In [12]:
# Recarregar CNES com todas as páginas
def buscar_todas_paginas(cod_municipio, nome):
    todos = []
    offset = 0
    while True:
        url = (f"https://apidadosabertos.saude.gov.br/cnes/estabelecimentos"
               f"?codigo_municipio={cod_municipio}&limit=20&offset={offset}")
        r = requests.get(url, timeout=15)
        lote = r.json().get('estabelecimentos', [])
        if not lote:
            break
        todos.extend(lote)
        offset += 20
    return pd.DataFrame(todos)

print("Buscando CNES...")
df_cuiaba_full = buscar_todas_paginas('510340', 'Cuiabá')
df_vg_full = buscar_todas_paginas('510840', 'Várzea Grande')
df_cnes_full = pd.concat([df_cuiaba_full, df_vg_full], ignore_index=True)

# Filtrar com coordenadas válidas
df_cnes_full['codigo_cnes'] = df_cnes_full['codigo_cnes'].astype(str).str.strip()
df_cnes_geo = df_cnes_full[
    df_cnes_full['latitude_estabelecimento_decimo_grau'].notna() &
    (df_cnes_full['latitude_estabelecimento_decimo_grau'] != 0)
].copy()

print(f"CNES com coordenadas: {len(df_cnes_geo)}")

# Cruzar carga histórica com CNES
df_score = carga_unidade.merge(
    df_cnes_geo[['codigo_cnes', 'nome_fantasia', 'bairro_estabelecimento',
                 'latitude_estabelecimento_decimo_grau',
                 'longitude_estabelecimento_decimo_grau',
                 'codigo_municipio', 'codigo_tipo_unidade']],
    left_on='ID_UNIDADE',
    right_on='codigo_cnes',
    how='inner'
)

print(f" Unidades cruzadas: {len(df_score)}")
print(f"\nTop 10 por carga histórica:")
cols = ['nome_fantasia', 'bairro_estabelecimento', 'casos_historicos', 
        'codigo_tipo_unidade']
print(df_score.nlargest(10, 'casos_historicos')[cols].to_string())

Buscando CNES...
CNES com coordenadas: 2941
 Unidades cruzadas: 158

Top 10 por carga histórica:
                                   nome_fantasia bairro_estabelecimento  casos_historicos  codigo_tipo_unidade
0  HOSPITAL E PRONTO SOCORRO MUNICIPAL DE CUIABA           BANDEIRANTES              7574                    5
1                             POLICLINICA VERDAO                 VERDAO              6997                    4
2   CENTRO DE ESPECIALIDADES MEDICAS DO PLANALTO                CARUMBE              4868                   36
3                        POLICLINICA DO PEDRA 90               PEDRA 90              3636                    4
4                                     CEM COXIPO                 COXIPO              3226                   36
5                      POLICLINICA PASCOAL RAMOS          PASCOAL RAMOS              2149                    4
6              HOSPITAL E MATERNIDADE SAO MATEUS        BOSQUE DA SAUDE              1732                    5
7              

In [13]:
# Score normalizado por unidade
max_casos = df_score['casos_historicos'].max()
df_score['score'] = df_score['casos_historicos'] / max_casos

# Classificação de risco
def classificar_risco(score):
    if score >= 0.75: return 'Alto'
    elif score >= 0.50: return 'Moderado-Alto'
    elif score >= 0.25: return 'Moderado'
    elif score >= 0.10: return 'Baixo'
    else: return 'Muito Baixo'

df_score['risco'] = df_score['score'].apply(classificar_risco)

print("Score calculado!")
print(f"\nDistribuição de risco:")
print(df_score['risco'].value_counts())

# Criar mapa Folium
import folium
from folium.plugins import HeatMap

# Centro entre Cuiabá e Várzea Grande
mapa = folium.Map(
    location=[-15.62, -56.09],
    zoom_start=12,
    tiles='CartoDB positron'
)

# Cores por risco
cores = {
    'Alto': '#d73027',
    'Moderado-Alto': '#fc8d59',
    'Moderado': '#fee090',
    'Baixo': '#91bfdb',
    'Muito Baixo': '#4575b4'
}

# Adicionar marcadores
for _, row in df_score.iterrows():
    lat = row['latitude_estabelecimento_decimo_grau']
    lon = row['longitude_estabelecimento_decimo_grau']
    cor = cores[row['risco']]
    
    folium.CircleMarker(
        location=[lat, lon],
        radius=6 + row['score'] * 20,
        color=cor,
        fill=True,
        fill_color=cor,
        fill_opacity=0.8,
        popup=folium.Popup(
            f"<b>{row['nome_fantasia']}</b><br>"
            f"Bairro: {row['bairro_estabelecimento']}<br>"
            f"Casos históricos: {row['casos_historicos']:,}<br>"
            f"Score: {row['score']:.2f}<br>"
            f"Risco: <b>{row['risco']}</b>",
            max_width=250
        ),
        tooltip=f"{row['nome_fantasia']} — {row['risco']}"
    ).add_to(mapa)

# Heatmap
heat_data = df_score[['latitude_estabelecimento_decimo_grau',
                       'longitude_estabelecimento_decimo_grau',
                       'casos_historicos']].values.tolist()
HeatMap(heat_data, radius=25, blur=15, min_opacity=0.4).add_to(mapa)

# Legenda
legenda = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000;
     background-color: white; padding: 15px; border-radius: 8px;
     border: 2px solid #ccc; font-size: 13px;">
<b>🦟 Score de Risco — Dengue MT</b><br>
<i>Carga histórica por unidade de saúde</i><br><br>
🔴 Alto (&gt;75%)<br>
🟠 Moderado-Alto (50–75%)<br>
🟡 Moderado (25–50%)<br>
🔵 Baixo (10–25%)<br>
⚫ Muito Baixo (&lt;10%)<br><br>
<small>Fonte: SINAN/DATASUS + CNES 2022–2024</small>
</div>
"""
mapa.get_root().html.add_child(folium.Element(legenda))

# Salvar
mapa.save('../reports/mapa_risco_dengue.html')
print("Mapa salvo em reports/mapa_risco_dengue.html")
print(f"\nTop 5 unidades de maior risco:")
print(df_score.nlargest(5, 'score')[
    ['nome_fantasia', 'bairro_estabelecimento', 'casos_historicos', 'risco']
].to_string())

Score calculado!

Distribuição de risco:
risco
Muito Baixo      146
Baixo              6
Moderado           3
Alto               2
Moderado-Alto      1
Name: count, dtype: int64
Mapa salvo em reports/mapa_risco_dengue.html

Top 5 unidades de maior risco:
                                   nome_fantasia bairro_estabelecimento  casos_historicos          risco
0  HOSPITAL E PRONTO SOCORRO MUNICIPAL DE CUIABA           BANDEIRANTES              7574           Alto
1                             POLICLINICA VERDAO                 VERDAO              6997           Alto
2   CENTRO DE ESPECIALIDADES MEDICAS DO PLANALTO                CARUMBE              4868  Moderado-Alto
3                        POLICLINICA DO PEDRA 90               PEDRA 90              3636       Moderado
4                                     CEM COXIPO                 COXIPO              3226       Moderado


In [14]:
import webbrowser
import os

caminho = os.path.abspath('../reports/mapa_risco_dengue.html')
webbrowser.open(f'file:///{caminho}')

True